## Objectives

After completing this lab you will be able to:

- Use FastMCP to create MCP Servers in either **STDIO** and **HTTP** transports
- Register custom tools, resources, and prompts to an MCP Server
- Test MCP Servers with Client connections and manual tool calling
- Create a Multi-server client and ReAct agent to use all our MCP tools


For this lab, we will be using the following libraries:

*   [`fastmcp`](https://gofastmcp.com/getting-started/welcome) for creating and configuring MCP servers and clients
*   [`langchain`](https://www.langchain.com/) for tooling convention and usage
*   [`langchain-mcp-adapters`](https://www.langchain.com/) for adapting MCP tools to be used in LangChain's ecosystem
*   [`langgraph`](https://www.langchain.com/) for creating agents using LangChain principles


In [1]:
import socket
import asyncio
from fastmcp import FastMCP, Client

In [2]:
import os
def make_dir():
    if os.path.exists("path"):
        print("✓ Path directory already exists")
    else:
        print("✗ Path directory doesn't exist - creating it...")
        os.makedirs("path")
        print("✓ Path directory created")

In [3]:
PORT = 8000

def test_port(port=PORT):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        try:
            s.bind(('127.0.0.1', port))
            return False
        except socket.error:
            return True

f"Port {PORT} is available: {not test_port()}"

'Port 8000 is available: True'

In [4]:
def print_stream_info(read, write, _sid, verbose=False):
    """Print information about the read/write streams and session ID.
    """
    if verbose:
        print("READ (receives FROM server):")
        print(read)
        print()
        
        print("WRITE (sends TO server):")
        print(write)
        print()
        
        print("SESSION ID:")
        print(_sid())

In [5]:
from langchain_core.tools import tool

This imports the  `@tool` decorator, which converts regular Python functions into **LangChain compatible** tools that work locally within a single application, To define a tool we simply write the decorator over a Python function. We can also access the tool information and call it using the `.invoke()` method.


In [6]:
@tool
def multiply(a: int, b: int) -> int:
   """Multiply two numbers."""
   return a * b

print(multiply.name)
print(multiply.description)
print(multiply.args)

print("What is 2 x 3?")
print("Answer: " + str(multiply.invoke({"a": 2, "b": 3})))

multiply
Multiply two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
What is 2 x 3?
Answer: 6


## Creating a Calculator MCPServer
MCP servers behave like functions that AI agents can call to perform actions using. Let's define and configure a basic MCP server.

- **FastMCP** creates the server instance with a name and instructions
- `@mcp.tool` decorator registers functions as callable tools
- `@mcp.prompt` decorator creates reusable prompt templates
- `@mcp.resource` decorator exposes data resources with URI patterns


This creates a complete MCP server named **CalculatorMCPServer** with mathematical tools (add, subtract), a document reading resource, and a code review prompt template.


### FastMCP Object 

The first step is to create an MCP server object ```mcp``` for the calculator. Like before, the object will be used to control the server. The parameters are:

- `name`: "CalculatorMCPServer" - the name of the server  
- `instructions`: "This server provides mathematical calculation tools. Call ```add()``` and ```subtract()``` to perform basic arithmetic operations." - this describes the functionality of the server


In [7]:
from fastmcp import FastMCP

mcp = FastMCP(
    name="CalculatorMCPServer",
    instructions="""
        This server provides data analysis tools.
        Call get_average() to analyze numerical data.
    """)
print('mcp object',mcp)

mcp object FastMCP('CalculatorMCPServer')


### Tools
 
 **Tools** (Active Capabilities)
    - Functions that AI agents can call to perform actions
    - Similar to LangChain tools but networked and discoverable

We define the MCP tools `add` and `subtract` - these will be called on the MCP object. 


In [8]:
@mcp.tool
def add(a: int, b: int) -> int:
    """
    Add two integers together.

    Args:
        a (int): The first integer.
        b (int): The second integer.

    Returns:
        int: The sum of `a` and `b`.

    Example:
        >>> add(3, 5)
        8
    """
    return a + b


@mcp.tool
def subtract(a: int, b: int) -> int:
    """
    Subtract one integer from another.

    Args:
        a (int): The number to subtract from.
        b (int): The number to subtract.

    Returns:
        int: The result of `a - b`.

    Example:
        >>> subtract(10, 4)
        6
    """
    return a - b



### Resources
Resources are like filing cabinets that AI systems can open to read information. Think of them as "files" or "data sources" that AI can easily access using simple web-like addresses for finding exactly what they need.

We create resources using the `@mcp.resource` decorator. When you set up something like `file:///endpoint/{name}`, you're basically creating an address system where **named path parameter** gets replaced with whatever file the AI wants to read.

The URI is purely for resource identification, name the URI path something that fits the type, style, and content of resource you are exposing.

Let's create two resources, one that returns a prewritten message template and another that returns the contents of an actual file.


In [9]:
@mcp.resource("file:///endpoint/{name}")
def return_template_document(name: str) -> str:
    """Read a document by name"""
    return f"Document contents of {name}"

In [ ]:
make_dir()

✗ Path directory doesn't exist - creating it...
✓ Path directory created


In [11]:
%%capture
!wget -P path/ https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/aNE__JjH4DLNEibuNpfDlg/examples.txt
!wget -P path/ https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/tfoeGPInNoajVS0DSohdVg/README.txt

In [12]:
@mcp.resource("file://endpoint2/{name}")
def read_document(name: str) -> str:
    """Read a document by name from the path directory"""
    try:
        # Read from the actual file system path
        with open(f"path/{name}", "r") as f:
            return f.read()
    except FileNotFoundError:
        return f"Document '{name}' not found in path directory"
    except Exception as e:
        return f"Error reading document: {str(e)}"

### Prompts 

Prompts are consistent, reusable templates that can be called for simple, repetitive tasks. They capture domain expertise in a structured way, so instead of reinventing instructions each time, the AI can rely on a proven pattern.


In [13]:
@mcp.prompt(title="Code Review")
def review_code(code: str) -> str:
    return f"Please review this code:\n\n{code}"

## Creating a Client - In-Memory Transport

Now that we've created our MCP server with tools, let's create a client to test it. For this first example, we'll use the simplest connection method: in-memory transport. When your server and client are running in the same Python process, you can connect directly without creating a separate transport object.
In a Pythonic sense, we'll use the client object to call the tools you need via methods. However, because it's actually communicating with a server, you will have to use several special Python functions to communicate with the server, such as async and await for **asynchronous operations** 


In [14]:
from fastmcp import  Client

client = Client(mcp)
print(f"client: {client}")

client: <fastmcp.client.client.Client object at 0x0000015AB08F81A0>


### Tools

We create a client to test our MCP server and call its tools remotely. Even though the server is running in the same process (in-memory transport), we still follow the MCP protocol pattern of client-server communication. The `call_tool()` method invokes a specific tool with parameters. When we call `client.call_tool` we are not really calling a local object, we are calling a server through the client

We wrap this in a function because we are dealing with a server that may take time to respond we will always use the following asynchronous functions :

- `async with client`: The server communication involves waiting time, so we use the async context manager which automatically manages the client lifecycle (opening and closing the connection)
- `await`: executes the async function and allows the Python process to wait for the server's response without blocking


In [15]:
async def call_add_tool(a: int, b: int):
    async with client:
        result = await client.call_tool("add", {"a": a, "b": b})
        return result

In [16]:
response = await call_add_tool(4, 5)
response

CallToolResult(content=[TextContent(type='text', text='9', annotations=None, meta=None)], structured_content={'result': 9}, meta={'fastmcp': {'wrap_result': True}}, data=9, is_error=False)

In [17]:
# The actual answer/data
print("\nResult Data .data :")
print(response.data)  # 9

# Content (text format)
print("\nContent (as text):")
print(response.content[0].text)  # "9"

# Structured content (as dictionary)
print("\nStructured Content:")
print(response.structured_content)  


Result Data .data :
9

Content (as text):
9

Structured Content:
{'result': 9}


In [18]:
## Question 
# Write a function to call the subtract tool with parameters a=4 and b=5, then execute it and print the result. What will be the output?<details>
#   <summary>Click here for hint</summary>

#   ```python
async def call_subtract_tool(a: int, b: int):
      async with client:
          result = await client.call_tool("subtract", {"a": a, "b": b})
          return result

response = await call_subtract_tool(4, 5)
print(response.content[0].text)


-1


In [19]:
async with client:
    tools = await client.list_tools()
    print("Available tools:")
    for tool in tools:
        print(f"- {tool.name}: {tool.description}")

Available tools:
- add: Add two integers together.
- subtract: Subtract one integer from another.


In [20]:
tool_obj = tools[0]
print(tool_obj)

name='add' title=None description='Add two integers together.' inputSchema={'additionalProperties': False, 'properties': {'a': {'type': 'integer', 'description': 'The first integer.'}, 'b': {'type': 'integer', 'description': 'The second integer.'}}, 'required': ['a', 'b'], 'type': 'object'} outputSchema={'properties': {'result': {'type': 'integer'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-wrap-result': True} icons=None annotations=None meta={'fastmcp': {'tags': []}} execution=None


In [21]:
input_schema = tool.inputSchema
print(input_schema)

{'additionalProperties': False, 'properties': {'a': {'type': 'integer', 'description': 'The number to subtract from.'}, 'b': {'type': 'integer', 'description': 'The number to subtract.'}}, 'required': ['a', 'b'], 'type': 'object'}


```Output Schema```: JSON structure defining what the tool returns (data type and structure of the response) - necessary because responses are transmitted over the network as JSON; tells clients (and LLMs) what to expect back. MCP doesn't always expose this since some tools perform actions (like saving to database) where the LLM only needs simple confirmation, not detailed output data.


In [22]:
output_schema = tool.outputSchema
print(output_schema )

{'properties': {'result': {'type': 'integer'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-wrap-result': True}


### Resources

We'll call the `read_resource` method in an asynchronous context manager wrapped in a function. The parameter for reading the resource is the URI with our configured `name`.


In [23]:
async def call_resource(name):
    async with client:
        result = await client.read_resource(f"file:///endpoint/{name}")
        return result

Use the `call_resource` function that wraps the async context manager to call the document contents here, it (`endpoint`) should return the template message.


In [24]:
response = await call_resource("README.txt")
print(response[0].text)

Document contents of README.txt


In [25]:
async def call_resource2(name):
    async with client:
        result = await client.read_resource(f"file://endpoint2/{name}")
        return result

This time when we input the parameter **README.txt**, it will retrieve the actual file from the disk which is referenced by `endpoint2/README.txt`.



In [26]:
response = await call_resource2("README.txt")


And if a requested file doesn't exist at `path` MCP servers typically handle file not found errors automatically and return error messages:


In [27]:
response = await call_resource2("random.txt")
resource = response[0]

In [28]:
print(f"uri:      {resource.uri}")
print(f"mimeType: {resource.mimeType}")
print(f"meta:     {resource.meta}")
print(f"text:     {resource.text}")

uri:      file://endpoint2/random.txt
mimeType: text/plain
meta:     None
text:     Document 'random.txt' not found in path directory


### Prompts

We'll also create a function to call the prompt method to review code. The only parameter is the code content to be reviewed.


In [29]:
async def call_prompt(code):
    async with client:

        result = await client.get_prompt("review_code", {"code": code})
        return result

In [30]:
response = await call_prompt("CODE TO BE REVIEWED")


In [31]:
message=response.messages[0]
print(f"Prompt Role:{message.role}")
print(f"Prompt Content:{message.content.text}")

Prompt Role:user
Prompt Content:Please review this code:

CODE TO BE REVIEWED


## HTTP Transport MCP Servers
HTTP transport allows MCP servers to run as web services that clients connect to via URLs. This is ideal for remote servers, cloud deployments, or when you want multiple clients to share the same server instance.


First, check if the port is available (True). If available, run the following cell to start the MCP server on that port. If the port is unavailable, change the PORT variable and rerun the server. This requirement is just for JupyterLab.


In [32]:
f"Port {PORT} is available: {not test_port()}"

'Port 8000 is available: True'

### Starting HTTP MCP Server

Let's launch the MCP server as an HTTP service running in the background. Using the `mcp` object, we call the method `mcp.run_http_async()` which starts the HTTP server on the specified port:
```python
mcp.run_http_async(port=PORT)
```

- Server runs continuously in the background
- `/mcp` endpoint is automatically created for JSON-RPC communication


for jupyter notebook only
However, because we are running this in a Jupyter notebook, we have to use `asyncio.create_task()`, which runs the server concurrently without blocking, allowing us to continue using other cells 


In [ ]:

#server_task = asyncio.create_task(server_task_)
asyncio.create_task(mcp.run_http_async(port=PORT))
print(f"HTTP MCP Server started in background on port {PORT}")

HTTP MCP Server started in background on port 8000




┌─────────────────────────────────────────────────────────────────────────────┐
│                                                                             │
│                                                                             │
│                        ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                        █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                             │
│                                                                             │
│                                                                             │
│                                FastMCP 3.4.3                                │
│                            https://gofastmcp.com                            │
│                                                                             │
│                 🖥  Server:      CalculatorMCPServer, 3.4.3                  │
│                 🚀 Deploy free: https

[07/06/26 15:50:08] INFO     Starting MCP server 'CalculatorMCPServer' with transport 'http' on    ]8;id=11601966;file://g:\compl_langchain\.venv\Lib\site-packages\fastmcp\server\mixins\transport.py\transport.py]8;;\:]8;id=11601967;file://g:\compl_langchain\.venv\Lib\site-packages\fastmcp\server\mixins\transport.py#330\330]8;;\
                             http://127.0.0.1:8000/mcp                                                             

INFO:     Started server process [12400]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:49184 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:49184 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:49184 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:49274 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49275 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:49276 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49277 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49278 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49279 - "DELETE /mcp HTTP/1.1" 200 OK


 **IMPORTANT NOTE**

If you want to **change** or **rerun** the server at any point, you **MUST** change the port as to not crash the kernel.


In [34]:
from fastmcp.client.transports import StdioTransport, StreamableHttpTransport
transport_http = StreamableHttpTransport(
    url=f"http://127.0.0.1:{PORT}/mcp")

In [35]:
http_client = Client(transport_http)
print('http_client',http_client )


http_client <fastmcp.client.client.Client object at 0x0000015AB0A8A850>


We define an async function `test_client_http` that takes a client object and two integers as parameters. The function uses an async context manager (`async with client`) to establish a connection to the MCP server, then calls the "add" tool with the provided arguments and returns the result, .


In [36]:
async def test_client_http(client: Client, a: int, b: int)->int:
    async with client:  
        result = await client.call_tool("add", {"a": a, "b": b})
        return result

We call the `test_client` function with our HTTP client and the values 4 and 5, awaiting the result. The response contains the tool's output, which we access by navigating to the first content item's text property and printing it.


In [37]:
response = await test_client_http(http_client, 4, 5)
print(response.content[0].text)

9
